# Train one HGT network with hierarchical skill family/direction and auxiliary intent

Run **26_prepare_skill_intent_target_relative_missile_nn_duel_skill_forecasting.ipynb**
first. This is the **single-seed, auxiliary-intent counterpart to notebook 25**.
Run All fits **one hierarchical network with seed 42**, selects its checkpoint using skill test
NLL, then saves skill and intent predictions, baselines and diagnostics.

The network retains notebook 25's HGT-v2 encoder: two width-64 graph layers with
four heads; separately gated aircraft/ownership/targeting families; 18 physical
edge features; and the learnable distance/CPA/bearing/energy threat prior.
The temporal Transformer remains width 96, six heads, two layers, dropout 0.25.
Inputs are unchanged 51-frame histories (115 channels for the full catalogue).
Missile positions and velocities are relative to their targets. Both aircraft's
backward accelerations and current own-skill indicators remain available.

The skill readout first predicts **29 families**, then **left/right conditioned on
each of the four mirrored families**. The 25 singleton families need no direction.
These distributions reconstruct probabilities for the original 33 skills:
`P(skill) = P(family) * P(direction | family)` for mirrored skills, or just
`P(family)` for singletons. The default prediction is the largest **joint skill
probability**, preserving the original evaluation task. Optional greedy decoding
can instead select the family first and then its direction.

An auxiliary MLP predicts **offensive, defensive, context_dependent** at each +1/+2/+5s
horizon, alongside the hierarchical skill readout. The three groups are frozen in the
prepared dataset, with 9/11/13 catalogue entries. Generic manoeuvres retain a
context-dependent label rather than being assigned an unobserved intent.
**Family-first partial-credit loss:** reward the correct family even when the side
is wrong, and add a bonus for the correct side. Defaults are `family_reward=1`
and `handedness_reward=1`. For a true mirrored skill,

`C = (a * P(true_family) + b * P(true_family, true_side)) / (a + b)`.

For a singleton, `C = P(true_family)`. Training minimizes
`L = leaf_weight * (-log C) + 0.25 * intent_CE + 1e-6 * L1`, averaged over valid
labels within each horizon and then over represented horizons. In the hard-decision
limit: correct family+side gets normalized credit 1, correct family with wrong side
gets 0.5, and a wrong family gets 0. Side correctness never earns a bonus without
the correct family. Non-directional skills have no side penalty or fake bonus.
The original leaf class weights apply to the entire per-example skill loss.
All heads share the encoder, so auxiliary supervision reaches the graph and
temporal layers. Future skill/intent labels are never encoder inputs.

The grouped 40/30/30 partitions, training-only scaling, non-overlapping training
histories and missing-label masks come from the base dataset without alteration.
Training retains AdamW, 5% learning-rate warm-up and cosine decay, class-weighted
skill CE, L1, and stopping checks after epoch 20. The skill head's **unweighted test
NLL alone selects checkpoints**; validation is evaluated only after training ends.
These are initial settings, not evidence of improvement over notebook 25.

The intent head is evaluated against both an intent training-frequency baseline
and probabilities obtained by grouping the trained skill head. This reveals
whether the separate head adds anything beyond the existing skill predictions.
One seed cannot estimate variability across independent fits.

Run All trains once and writes `best.pt`, resumable `last.pt`, both sets of final
prediction NPZs, metric CSVs and diagnostic PNGs. Install `.[ml]` and a Jupyter/
IPython kernel. CPU uses FP32; CUDA can use BF16 training with FP32 graph reductions
and FP32/math evaluation. Every code cell is timed.

Artifacts use `artifacts/nn_duel_skill_intent_forecasting/`, with a singular
`comparisons/latest_hgt_hierarchical_intent_run.json` receipt. Set `BVR_NN_INTENT_DATA` to override
the prepared data directory or `BVR_NN_INTENT_MLFLOW=0` to disable optional tracking.
For resume/report-only execution, supply the one `EXISTING_RUN` path. Source, data,
taxonomy and config checks protect reuse. Start fresh runs for this architecture;
earlier flat-head checkpoints have a different skill readout.

Notebook 26 must provide the new **v2 label overlay**, including the frozen family/
direction mapping. Re-run notebook 26 to refresh an earlier intent-only overlay;
no new base features, scaling or scenario splits are required. Family metrics and
conditional-direction metrics are saved separately. Direction reports condition
on the true paired family; they are not end-to-end original-skill accuracy.

The reward is optimized through smooth probabilities, not through a hard argmax
or reinforcement-learning update. For the default rewards the extra handedness
penalty is bounded by `log(2)`, so a confidently wrong side cannot dominate family
recognition. The joint 33-skill NLL is still reported and selects checkpoints, but
it is no longer the skill training objective. `family_reward` controls base family
credit and `handedness_reward` the additional side bonus; setting the latter to zero
trains the family task only. Existing notebook-26 v2 labels remain suitable; start
fresh notebook-27 runs with the new objective. No re-preparation is needed.


In [ ]:
%%time
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_NN_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "bvr_behavior_prediction/nn_duel_skill_forecasting.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the kernel environment and native-library repair steps.
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from bvr_behavior_prediction import nn_duel_skill_forecasting as duel_data
from bvr_behavior_prediction import skill_intent_forecasting as intent_data
from bvr_behavior_prediction.paired_skill_forecasting import (
    HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS, read_json, write_json, sha256,
)
DATA_ROOT = ROOT / "artifacts/nn_duel_skill_intent_forecasting"
EXPERIMENT_ROOT = ROOT / "artifacts/nn_duel_skill_intent_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

import torch
from bvr_behavior_prediction.paired_skill_forecasting import (
    DEFAULT_TRAINING, train_model, final_evaluation, metric_tables, resolve_precision,
    fit_class_weighting, class_weight_table,
)
from bvr_behavior_prediction.paired_skill_tracking import log_run_artifacts
from bvr_behavior_prediction.enemy_missile_diagnostics import save_diagnostics

DATA_OVERRIDE = os.environ.get("BVR_NN_INTENT_DATA")
if not DATA_OVERRIDE and not (DATA_ROOT / "latest_dataset.json").exists():
    raise FileNotFoundError("Run notebook 26 first, or set BVR_NN_INTENT_DATA to its dataset directory.")
receipt = None if DATA_OVERRIDE else read_json(DATA_ROOT / "latest_dataset.json")
DATA = Path(DATA_OVERRIDE or receipt["directory"]).expanduser().resolve()
if receipt and sha256(DATA / "manifest.json") != receipt["manifest_sha256"]:
    raise ValueError("The prepared dataset changed since notebook 26 completed.")
data = intent_data.IntentNNDuelData(DATA)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42  # Exactly one training seed and one fitted network.
ARCHITECTURE = "hgt_hierarchical_intent"
TRAIN_MODEL = True
REEVALUATE_SAVED = False
RESUME = False
EXISTING_RUN = None  # Path required for resume or report-only execution.
if (RESUME or not TRAIN_MODEL) and not EXISTING_RUN:
    raise ValueError("Set EXISTING_RUN when resuming or reporting.")
if EXISTING_RUN and TRAIN_MODEL and not RESUME:
    raise ValueError("Use RESUME=True for an existing run, or clear EXISTING_RUN for a fresh fit.")
RUN_STAMP = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
RUN = (Path(EXISTING_RUN).expanduser().resolve() if EXISTING_RUN else
       EXPERIMENT_ROOT / "runs" / f"nn-duel-hierarchical-skill-intent-hgt-s{SEED}_{RUN_STAMP}")
CONFIG = dict(DEFAULT_TRAINING)
CONFIG.update(architecture=ARCHITECTURE, seed=SEED, intent_loss_weight=0.25,
              hierarchical_loss="partial_credit", family_reward=1.0, handedness_reward=1.0, observer_acceleration=True, target_relative_missiles=True,
              target_relative_missile_velocities=True, width=96, heads=6, layers=2,
              graph_width=64, graph_layers=2, graph_heads=4, hgt_version=2,
              threat_cpa_horizon_s=30.0, threat_time_scale_s=5.0, threat_prior_strength=1.0,
              missile_distance_floor_m=1000.0, missile_distance_power=1.0,
              dropout=0.25, epochs=100, learning_rate=3e-4,
              lr_schedule="warmup_cosine", warmup_fraction=0.05,
              warmup_start_ratio=0.1, min_lr_ratio=0.05,
              early_stopping=True, early_stopping_min_epochs=20,
              early_stopping_patience=10, l1_lambda=1e-6,
              class_weighting="sqrt_inverse", max_class_weight_ratio=5.0,
              overtraining_shutdown=True, overtraining_min_epochs=20, overtraining_patience=3,
              overtraining_relative_increase=0.05, overtraining_min_gap=0.1,
              precision="bf16", evaluation_precision="fp32", evaluation_attention="math",
              batch_size=512,
              evaluation_batch_size=1024, data_backend="prefetch" if DEVICE == "cuda" else "direct",
              num_workers=2, cpu_threads=4, compile_model=False)
MLFLOW_ENABLED = os.environ.get("BVR_NN_INTENT_MLFLOW", "1") == "1"
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI",
    f"sqlite:///{(EXPERIMENT_ROOT / 'mlflow.db').as_posix()}")
TRACKING = {"enabled": MLFLOW_ENABLED, "tracking_uri": MLFLOW_TRACKING_URI,
            "experiment": "nn-duel-skill-intent-forecasting", "log_per_class": True,
            "log_model": True,
            "artifact_root": (EXPERIMENT_ROOT / "mlflow_artifacts").as_uri()
                             if MLFLOW_TRACKING_URI.startswith("sqlite:") else None}
print("Prepared dataset:", DATA)
print("Single model run:", RUN)
display(pd.Series({**CONFIG, "device": DEVICE,
    "effective_precision": resolve_precision(DEVICE, CONFIG["precision"]),
    "input_features": len(data.manifest["features"]), "classes": len(data.classes),
    "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS}))
display(pd.DataFrame(data.manifest["counts"]).T)
if MLFLOW_ENABLED and MLFLOW_TRACKING_URI.startswith("sqlite:"):
    print(f'mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 127.0.0.1 --port 5000')
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.synchronize()

# This audit reads only labels retained in the training partition.
CLASS_WEIGHTING = fit_class_weighting(data, CONFIG)
CLASS_WEIGHT_AUDIT = class_weight_table(CLASS_WEIGHTING)
with pd.option_context("display.max_rows", None):
    display(CLASS_WEIGHT_AUDIT.pivot(index="skill", columns="horizon_s",
                                    values=["training_support", "weight"]).round(4))
print("Zero-training-support skill/horizon pairs:")
display(CLASS_WEIGHT_AUDIT.loc[CLASS_WEIGHT_AUDIT.training_support.eq(0)])

print("Intent class order:", data.intent_classes)
print("Auxiliary intent loss weight:", CONFIG["intent_loss_weight"])
display(pd.read_csv(DATA / "intent_support_by_split.csv"))

print("Family count:", len(data.family_classes))
print("Conditional-direction families:", data.manifest["skill_hierarchy"]["paired_families"])
print("Direction order:", data.direction_classes)
display(pd.read_csv(DATA / "direction_support_by_split.csv"))

print("Skill objective:", CONFIG["hierarchical_loss"])
print("Family credit:", CONFIG["family_reward"], "| additional handedness bonus:", CONFIG["handedness_reward"])


## Fit one network: family, conditional direction and auxiliary intent

Let `n[h,c]` be the number of retained training labels for class `c` at horizon
`h`, ignoring missing targets (`-1`), and `n_max[h]` the largest such class count.
For classes seen during training, form

`r[h,c] = min(5, sqrt(n_max[h] / n[h,c]))`

and normalise by the training-frequency weighted mean:

`w[h,c] = r[h,c] / sum_c(training_frequency[h,c] * r[h,c])`.

The rare/common per-example loss ratio is at most 5. For example, an uncapped 16:1
support ratio gives rare examples four times the weight of common examples. The
frequency-weighted mean weight is one for each horizon, keeping the expected loss
scale comparable with unweighted CE and L1. Zero-support classes receive zero
positive-target weight and retain their output logits. Their absence is audited.

For each horizon, average `w[h,y] * (-log C[h,y])` over its valid labels; then average
the represented horizons equally. The denominator is the number of valid labels,
not the minibatch sum of weights: weights remain effective even when a minibatch
contains just one class. Missing labels contribute neither loss nor gradient.
The masked partial-credit loss is implemented in FP32 under BF16 training too.

The loss uses independent missing-label masks. Scaling and the split are inherited unchanged through notebook 26.
The training objective adds `l1_lambda * sum(abs(model_weights))` (default `1e-6`) to
the class-weighted partial-credit skill loss plus the auxiliary intent CE. This includes trainable matrix/tensor weights and position
embeddings, excluding biases and LayerNorm scales. AdamW weight decay remains 0.01.
Train/test NLL is evaluated without L1, in evaluation mode; checkpoint selection
therefore remains comparable across epochs. History records the L1 contribution,
evaluation-mode joint training data loss (`train_data_loss`), its skill component
(`train_skill_data_loss`) and unweighted intent NLL (`train_intent_nll`), its sum with
L1 (`train_objective`), and the unweighted test-minus-train NLL gap separately.
Unweighted mean test NLL still selects checkpoints and drives both stopping rules;
the objective is not changed to favour a checkpoint based on weighted test scores.

The warm-up/cosine schedule is based on the planned number of optimizer updates,
not observed validation metrics. Its state, current optimizer learning rate and
completed-update count are saved together. Exact resume requires unchanged data,
source, epoch budget, batch size and schedule settings. Epoch history records the
first/last used learning rate, next-update rate and cumulative optimizer steps.

Stopping checks begin **after epoch 20** for this recipe. Ten stale test-NLL epochs
trigger ordinary early stopping. The independent overtraining rule requires three
consecutive epochs where test NLL exceeds its best by more than 5%, test-minus-train
NLL exceeds 0.1, and train NLL is below its value at the best-test checkpoint.
A recovery resets its counter. Checkpoint selection runs from epoch one, and both
rules preserve `best.pt`. Set both stopping switches to False to complete the full
cosine schedule. These are starting settings, not tuned performance guarantees.
Checkpoint input dimensions, feature ordering and dataset digest are stored with weights.

**Evaluation policy:** training can use CUDA BF16, while checkpoint selection and
final evaluation explicitly use `evaluation_precision="fp32"` and
`evaluation_attention="math"`. This keeps FP32 cross-entropy and unweighted NLL,
and isolates evaluation from CUDA BF16 and fused inference-attention kernels.
This is a conservative evaluation policy, not a confirmed diagnosis of the reported
GPU failure. It matches notebook 25. Persistent failures write
`nonfinite_evaluation.json` in the run directory with the processing stage and
sample details; do not replace nonfinite values or discard those examples.

Start fresh timestamped runs for this schema. Run notebook 26 to prepare the
intent overlay from notebook-24 v2 data. Notebook-25 checkpoints lack the intent head. Resume
requires the same source, prepared dataset, code and configuration. Reuse completed
final outputs only with `TRAIN_MODEL=False` and `REEVALUATE_SAVED=False`.

The skill objective is now a **family-first partial-credit loss**, not the original
joint skill cross-entropy. Write `p_f = P(true family)` and
`p_d = P(true side | true family)`. For a mirrored skill, the normalized credit is

`C = (a * p_f + b * p_f * p_d) / (a + b)`.

For singleton skills, `C = p_f`. The starting values are `a=b=1`, so the limiting
credit is 1 for correct family+side, 0.5 for correct family with the opposite side,
and 0 for a wrong family. The skill loss is `-log C`, weighted by the original
leaf's class weight. Both the base family reward and the handedness bonus require
the correct family; a correct side in the wrong family cannot earn extra credit.

This is computed stably as `CE_family + log(a+b) - log(a + b * exp(-CE_direction))`
for mirrored labels, and `CE_family` for singleton labels. With equal rewards,
the extra side penalty lies between 0 and `log(2)`. Thus a wrong side keeps useful
family credit instead of incurring an unbounded direction CE. A nonnegative side
bonus is configurable; zero removes side supervision. The true family selects the
side branch during training, without a hard predicted-family decision.

Missing labels remain masked in both branches. Singleton direction labels remain
masked and do not receive an artificial third direction. The combined per-example
loss is averaged over **all valid skill labels** within each horizon, then across
represented horizons. Class weights multiply the whole skill loss. The total
objective remains `L_data + L1`, with
`L_data = L_skill_partial_credit + 0.25 * L_intent_unweighted_CE`.

The continuous expected credit and the discrete family-first credit are logged in
training history and `final_hierarchical_credit_metrics.csv`. Discrete credit uses
the family argmax followed by that family's side argmax; these reward metrics are
separate from joint-leaf accuracy. Unweighted **joint original-skill test NLL**
still selects checkpoints and drives stopping, preserving the reporting criterion
used in earlier runs. It is not presented as the modified training loss. Changing
the loss/reward settings requires a fresh run; optimizer/scheduler resume guards
include them. Notebook 26's v2 labels and masks need no change.

The default forward returns original-skill **log probabilities** `(batch,3,33)`;
applying softmax still yields the original-skill distribution. Request
`return_hierarchy=True` to obtain family logits `(batch,3,29)`, conditional direction
logits `(batch,3,4,2)`, joint skill log probabilities and auxiliary intent logits.
All heads are computed from one shared encoder pass. `return_intent=True` retains
the two-output skill/intent interface for callers that do not need the hierarchy.

This notebook fits **exactly one network, once, with SEED=42**. There is no seed
loop or ensemble. Resume requires the same v2 overlay, family/side ordering, code,
configuration and optimizer/scheduler state; earlier flat-head runs cannot resume
as this architecture.


In [ ]:
%%time
def epoch_output(row, summary, per_class):
    print(f"Epoch {row['epoch']}: train NLL={row['train_nll']:.5f}, "
          f"test NLL={row['test_nll']:.5f}, "
          f"train data loss={row['train_data_loss']:.5f}, "
          f"intent NLL={row['train_intent_nll']:.5f}, "
          f"family NLL={row['train_family_nll']:.5f}, "
          f"test credit={row['test_hierarchical_credit']:.3f}, "
          f"L1={row['train_l1_penalty']:.5f}, "
          f"gap={row['generalization_gap']:.5f}, "
          f"LR={row['learning_rate']:.3g}, "
          f"overtraining streak={row['overtraining_epochs']}, {row['seconds']:.1f}s", flush=True)
    display(summary[["split", "horizon_s", "nll", "accuracy", "macro_f1",
                     "supported_macro_f1", "weighted_f1"]].round(4))
    with pd.option_context("display.max_rows", None):
        display(per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                 values=["class_nll", "f1", "support"]).round(4))

tracking = {**TRACKING, "run_name": RUN.name}
if TRAIN_MODEL:
    history = train_model(data, RUN, CONFIG, callback=epoch_output, device=DEVICE,
                          resume=RESUME, tracking=tracking)
else:
    history = pd.read_csv(RUN / "history.csv")
status = read_json(RUN / "training_status.json")
if status["status"] not in {"completed", "early_stopped"}:
    raise ValueError("Training has not concluded.")
saved_run = read_json(RUN / "run.json")
if saved_run["data_manifest_sha256"] != sha256(DATA / "manifest.json"):
    raise ValueError("Saved run belongs to another prepared dataset.")
if (saved_run["classes"] != data.classes or saved_run["feature_names"] != data.manifest["features"]
        or saved_run["intent_supervision"] != data.manifest["intent_supervision"]
        or saved_run["skill_hierarchy"] != data.manifest["skill_hierarchy"]):
    raise ValueError("Saved catalogue, feature order or intent taxonomy differs.")
if saved_run["training"] != CONFIG:
    raise ValueError("Requested configuration differs from the saved run.")
display(history.tail())
if DEVICE == "cuda":
    torch.cuda.synchronize()


## Final evaluation and persisted reports

Training must finish before this cell accesses validation. Completed skill and
intent predictions are reused unless `REEVALUATE_SAVED=True`. No additional seed
is fitted. The completed-run receipt is written only after all reports finish.


In [ ]:
%%time
required_finals = ["final_metrics.csv", "final_per_class.csv", "selection.json",
                   "final_intent_metrics.csv", "final_intent_per_class.csv",
                   "final_family_metrics.csv", "final_family_per_class.csv",
                   "final_direction_metrics.csv", "final_direction_per_class.csv", "final_hierarchical_credit_metrics.csv",
                   *[f"final_family_{split}.npz" for split in SPLITS],
                   *[f"final_direction_{split}.npz" for split in SPLITS],
                   *[f"final_{split}.npz" for split in SPLITS],
                   *[f"final_intent_{split}.npz" for split in SPLITS]]
if not REEVALUATE_SAVED and all((RUN / name).is_file() for name in required_finals):
    best_epoch = read_json(RUN / "selection.json")["epoch"]
    summary = pd.read_csv(RUN / "final_metrics.csv")
    print("Reusing saved final predictions:", RUN)
else:
    best_epoch, summary, scores, results = final_evaluation(data, RUN, DEVICE, tracking=tracking)
    del results
display(Markdown(f"**Single seed {SEED}, skill-test-selected epoch {best_epoch}**"))
display(summary.round(4))
display(pd.read_csv(RUN / "final_intent_metrics.csv").round(4))
display(pd.read_csv(RUN / "final_family_metrics.csv").round(4))
print("Normalized family credit plus handedness bonus:")
display(pd.read_csv(RUN / "final_hierarchical_credit_metrics.csv").round(4))
print("Direction metrics condition on the true mirrored family, excluding singletons:")
display(pd.read_csv(RUN / "final_direction_metrics.csv").round(4))

# Both baselines use only retained training labels; no held-out labels fit probabilities.
for task, labels, classes in (("skill", data.targets, data.classes),
                               ("intent", data.intent_targets, data.intent_classes)):
    frequency = np.zeros((len(HORIZONS), len(classes)), np.float32)
    for h in range(len(HORIZONS)):
        selected = labels[data.indices["train"], h]
        counts = np.bincount(selected[selected >= 0], minlength=len(classes))
        frequency[h] = (counts + 1) / (counts.sum() + len(classes))
    rows = []
    for split in SPLITS:
        targets = labels[data.indices[split]]
        probabilities = np.broadcast_to(frequency, (len(targets), *frequency.shape))
        baseline, _, _ = metric_tables(targets, probabilities, classes, split)
        rows.append(baseline)
    baseline = pd.concat(rows, ignore_index=True)
    filename = "training_frequency_baseline.csv" if task == "skill" else "training_frequency_intent_baseline.csv"
    baseline.to_csv(RUN / filename, index=False)
    display(baseline.assign(task=task, model="Training frequency").round(4))

save_diagnostics(RUN, data=data)
intent_comparison = intent_data.save_intent_diagnostics(RUN, data)
display(intent_comparison.round(4))
if MLFLOW_ENABLED:
    log_run_artifacts(RUN)
receipt_path = EXPERIMENT_ROOT / "comparisons/latest_hgt_hierarchical_intent_run.json"
receipt_path.parent.mkdir(parents=True, exist_ok=True)
write_json(receipt_path, {
    "architecture": ARCHITECTURE, "run": str(RUN), "seed": SEED,
    "data_manifest_sha256": sha256(DATA / "manifest.json"),
    "configuration": CONFIG, "intent_supervision": data.manifest["intent_supervision"],
    "skill_hierarchy": data.manifest["skill_hierarchy"],
    "hierarchical_objective": saved_run["hierarchical_objective"],
})
print("Trained checkpoint:", RUN / "best.pt")
print("Completed single-run receipt:", receipt_path)


## Inspect the single run

`diagnostics/` contains skill reports. `intent_diagnostics/` compares the intent head
and grouped skill probabilities, including calibration, per-class tables and
confusion plots for every partition. `final_intent_*.npz` retain exact aligned
sample IDs, targets, probabilities and per-example logit cross-entropies.

`final_family_*.npz` and `final_family_metrics.csv` report the family task.
`final_direction_*.npz` and `final_direction_metrics.csv` report direction **given
the true paired family**, masking all singletons and missing labels. Empty
horizons retain count zero and undefined NLL/accuracy. The primary 33-class reports
use joint skill probabilities and include errors from either decision level.

`final_hierarchical_credit_metrics.csv` reports family accuracy, realized normalized
family-first credit, continuous expected credit, and unweighted partial-credit
log loss per split/horizon. With the default rewards, correct family/wrong side
gets half credit, while correct family/correct side gets full credit.


In [ ]:
%%time
from IPython.display import Image

display(pd.read_csv(RUN / "intent_diagnostics/summary.csv").round(4))
display(Image(filename=str(RUN / "diagnostics/learning_curves.png")))
display(Image(filename=str(RUN / "intent_diagnostics/confusion_intent_head_validation.png")))
print("Network weights:", RUN / "best.pt")


## Use the hierarchical network

`best.pt` contains the HGT/temporal encoder, family head, conditional direction
head, auxiliary intent head, normalization and frozen mappings. Reconstruct with
`make_model(len(saved["classes"]), saved["model_config"], saved["mean"], saved["std"])`,
load `saved["model_state"]` and call `eval()`.

For a normalized `(batch,51,115)` history:

```python
with torch.inference_mode():
    outputs = model(x, return_hierarchy=True)
    family_probabilities = outputs["family_logits"].softmax(-1)       # (batch,3,29)
    direction_probabilities = outputs["direction_logits"].softmax(-1) # (batch,3,4,2)
    skill_probabilities = outputs["skill_log_probs"].exp()           # (batch,3,33)
    joint_skill = skill_probabilities.argmax(-1)
    greedy = model.decode_hierarchy(outputs["family_logits"], outputs["direction_logits"])
```

`joint_skill` is used by the reported original-skill metrics. `greedy` contains
family-first `family_indices`, then `direction_indices` (-1 for a singleton),
and the corresponding original `skill_indices`. These decisions can differ when
a high-probability mirrored family splits its probability between left and right.

The class/family/paired-family/side order is in `saved["skill_hierarchy"]` and the
model configuration. Side order is **left, right**. Intent order is in
`saved["intent_supervision"]["classes"]`. Inputs still contain only the observer's
current skill; no future labels or predicted opponent family are encoder inputs.

The default `model(x)` / MLflow export returns normalized joint original-skill log
probabilities. Softmax remains compatible with the old skill-score interface.
Use the checkpoint factory and `return_hierarchy=True` to access every head.
`return_intent=True` returns the joint skill log probabilities and intent logits.
Training/resume requires the pinned base dataset and v2 label overlay; inference
from checkpoint weights does not read dataset files.
